In [1]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from feature_preprocessing import FeaturePreprocesser

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_validate
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, confusion_matrix

from tqdm import tqdm

In [2]:

import importlib
import feature_preprocessing

importlib.reload(feature_preprocessing)


<module 'feature_preprocessing' from 'C:\\Users\\WZ3H3LW\\Desktop\\szakdolgozat\\feature_preprocessing.py'>

In [7]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

columns_to_exclude = ['Gear_ID','Date']
y = train_df['Label']
y_val = test_df['Label']

rows = []
for _, row in tqdm(train_df.iterrows()):
    row_df = pd.DataFrame([row])
    processed = FeaturePreprocesser().process(row_df, 400, 16)
    rows.append(processed)

train_df = pd.concat(rows, ignore_index=True)

rows = []
for _, row in tqdm(test_df.iterrows()):
    row_df = pd.DataFrame([row])
    processed = FeaturePreprocesser().process(row_df, 400, 16)
    rows.append(processed)

test_df = pd.concat(rows, ignore_index=True)

X = train_df.drop(columns=columns_to_exclude)
X_val = test_df.drop(columns=columns_to_exclude)


scaler = MinMaxScaler()
X = scaler.fit_transform(X)
X_val = scaler.transform(X_val)

80it [00:22,  3.58it/s]
20it [00:05,  3.62it/s]


In [17]:
test_df

,num_of_peaks,num_of_troughs,max_num_of_peaks,max_num_of_troughs,min_num_of_peaks,min_num_of_troughs,max_peak,max_trough,min_peak,min_troughs,max_area_between,min_area_between,avarage_area_between,avarage_extra_material,min_extra_material,max_extra_material,Gear_ID,Date
0,2111.0,2109.0,140.0,140.0,125.0,125.0,0.696933,0.519165,-0.407901,-0.654505,104.446150,93.821948,100.204499,0.263453,0.004264,0.544943,example_ID_0,example_Date__0
1,2157.0,2155.0,141.0,141.0,126.0,125.0,0.730466,0.481428,-0.459452,-0.651587,104.559219,93.819979,99.713711,0.182551,0.009415,0.694406,example_ID_1,example_Date__1
2,2132.0,2130.0,142.0,142.0,126.0,127.0,0.731438,0.542561,-0.378812,-0.648731,105.981019,90.603265,99.932306,0.210098,0.024251,0.570043,example_ID_2,example_Date__2
3,2108.0,2103.0,137.0,136.0,121.0,122.0,0.717919,0.440633,-0.441455,-0.632991,106.815486,96.458611,102.123366,0.260588,0.066348,0.468577,example_ID_3,example_Date__3
4,2126.0,2121.0,140.0,140.0,126.0,125.0,0.707288,0.485769,-0.485690,-0.681217,106.984970,97.703838,101.694253,0.219294,0.030053,0.501211,example_ID_4,example_Date__4
5,2140.0,2140.0,140.0,139.0,127.0,128.0,0.755452,0.437166,-0.503169,-0.656765,107.151492,97.960104,101.668644,0.201462,0.046209,0.590394,example_ID_5,example_Date__5
6,2135.0,2132.0,139.0,140.0,128.0,127.0,0.719994,0.480477,-0.435633,-0.693712,105.834909,95.492723,100.496341,0.224506,0.022010,0.443260,example_ID_6,example_Date__6
7,2150.0,2153.0,141.0,142.0,126.0,126.0,0.717887,0.421234,-0.382600,-0.682384,108.154021,96.463439,101.189959,0.257729,0.037851,0.446493,example_ID_7,example_Date__7
8,2149.0,2146.0,142.0,142.0,129.0,128.0,0.678915,0.405078,-0.443939,-0.633532,105.563462,97.334024,101.493589,0.174169,0.011997,0.531786,example_ID_8,example_Date__8
9,2141.0,2139.0,138.0,138.0,128.0,128.0,0.766216,0.540496,-0.434664,-0.623957,107.467859,93.268993,100.767876,0.210927,0.005369,0.627900,example_ID_9,example_Date__9


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y        
)

In [11]:
VotingCL = VotingClassifier(
        estimators=[
        ('logr', LogisticRegression(random_state=42)),
        ('gb', GradientBoostingClassifier(random_state=42)),
        ('hg', HistGradientBoostingClassifier(learning_rate=0.05,max_depth=3,random_state=42))
    ],
    voting='soft'
)

In [12]:
scores = cross_validate(
    VotingCL,
    X,
    y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring=["accuracy", "balanced_accuracy"]
)

In [13]:
print(f"Accuracy (CV átlag): {scores["test_accuracy"].mean():.3f} ± {scores["test_accuracy"].std():.3f}")
print(f"Balanced accuracy (CV átlag):{scores["test_balanced_accuracy"].mean():.3f} ± {scores["test_balanced_accuracy"].std():.3f}")
VotingCL.fit(X_train,y_train)

Accuracy (CV átlag): 1.000 ± 0.000
Balanced accuracy (CV átlag):1.000 ± 0.000


,estimators,"[('logr', ...), ('gb', ...), ...]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [18]:
y_pred = VotingCL.predict(X_val)
acc = accuracy_score(y_val, y_pred)
bal = balanced_accuracy_score(y_val, y_pred)
conf = VotingCL.predict_proba(X_val)
y_score = VotingCL.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_score)
print("Accuracy:", acc)
print("Balanced Accuracy:", bal)
print("AUC:", auc)
cm = confusion_matrix(y_val, y_pred)
print(cm)
print(f"False positive rate: {cm[0][1] / (cm[0][1] + cm[0][0]) * 100:.2f}%")
print(f"True negative rate: {cm[1][1] / (cm[1][0] + cm[1][1]) * 100.:2f}%")

Accuracy: 1.0
Balanced Accuracy: 1.0
AUC: 1.0
[[10  0]
 [ 0 10]]
False positive rate: 0.00%
True negative rate: 100.000000%


In [15]:
import joblib

In [16]:
joblib.dump(VotingCL, 'VotingCL.pkl')
joblib.dump(scaler, 'MinMaxScaler.pkl')

['MinMaxScaler.pkl']